# Observe the Infrastructure

The agent is running end-to-end. This notebook walks through the observability surfaces the
infrastructure exposes: runtime logs, GenAI traces, gateway audit logs, and AgentCore Memory.

You will need to have run at least one prompt through the agent (notebook 05) before the queries
below return data.

> **Demo logging:** the cells below print raw log events and memory records to make the
> workshop observable. Production systems should log metadata only — never full model
> responses or conversation content.


## Step 1: Open CloudWatch Logs

AgentCore Runtime streams logs to CloudWatch via OpenTelemetry. Open the log group named
`/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent-*-DEFAULT` in the console, or query it
from Python below.

Key log lines to look for:

| Log Entry | What It Confirms |
|-----------|-----------------|
| `[MODEL] Using LiteLLM Gateway: https://...` | Model calls routed through Module 2 |
| `[GATEWAY] URL: https://...` | Tools served by Module 3's gateway |
| `Getting OAuth2 token...` | M2M auth via Token Vault |
| `Negotiated protocol version: 2025-03-26` | MCP handshake with the gateway succeeded |
| `Created session: ...` | AgentCore Memory session created |
| `Started code interpreter in ...` | Code Interpreter sandbox activated |


In [ ]:
import json
import os
import time
import boto3

REGION = (
    os.environ.get("AWS_REGION")
    or os.environ.get("AWS_DEFAULT_REGION")
    or boto3.session.Session().region_name
)
if not REGION:
    raise RuntimeError(
        "No AWS region configured. Set AWS_REGION (or AWS_DEFAULT_REGION), "
        "or run `aws configure set region <region>`."
    )
logs = boto3.client("logs", region_name=REGION)


def recent_events(log_group, limit, hours_back=2, **extra):
    """Events from the last `hours_back` hours of a log group.

    Always pass startTime: without it FilterLogEvents scans from the group's beginning
    and can return an empty first page with a nextToken, which looks like "no events".
    Follow the token until events arrive or the scan ends.
    """
    kwargs = dict(
        logGroupName=log_group,
        limit=limit,
        startTime=int((time.time() - hours_back * 3600) * 1000),
        **extra,
    )
    for _ in range(20):
        page = logs.filter_log_events(**kwargs)
        if page.get("events") or not page.get("nextToken"):
            return page.get("events", [])
        kwargs["nextToken"] = page["nextToken"]
    return []


RUNTIME_LOG_GROUP_PREFIX = "/aws/bedrock-agentcore/runtimes/FAST_stack_FASTAgent"

matches = logs.describe_log_groups(logGroupNamePrefix=RUNTIME_LOG_GROUP_PREFIX).get("logGroups", [])
if not matches:
    print(f"No log groups matching {RUNTIME_LOG_GROUP_PREFIX}* found — invoke the agent first")
else:
    runtime_group = matches[0]["logGroupName"]
    print(f"Runtime log group: {runtime_group}")

    # Pull the key lines the table above describes (any of these terms), not the
    # instrumentation warnings that open every container start.
    KEY_LINES = '?"[MODEL]" ?"[GATEWAY]" ?"OAuth2 token" ?"Negotiated protocol version" ?"Created session"'
    events = recent_events(runtime_group, 10, filterPattern=KEY_LINES)
    print(f"{len(events)} key log lines from the last 2 hours (oldest first):")
    # Runtime logs carry the agent's reasoning and the end user's prompt
    # text. Notebook output is saved to the .ipynb on disk, so print the
    # shape of each record and its first line rather than the payload.
    for e in events[-10:]:
        # OpenTelemetry records put the log line in "body"; show that when present.
        try:
            body = json.loads(e["message"]).get("body")
        except (ValueError, AttributeError):
            body = None
        text = str(body) if body else e["message"]
        first_line = text.splitlines()[0][:80] if text.strip() else "(empty)"
        print(f"  {e['timestamp']}  {len(e['message']):5d} chars  {first_line}")
    print("\nFull messages: open the log group in the CloudWatch console.")

## Step 2: Inspect the Gateway Audit Log

Whichever gateway you wired, its request interceptor Lambda writes a structured audit record for
every tool call. The two gateways use different interceptor names, so the cell checks both log
groups and reports the one your agent is using: Module 3a's Tools Gateway logs to
`/aws/lambda/agentcore-gateway-request-interceptor`, Module 3b's AgentCore Gateway to
`/aws/lambda/ac-gateway-request-interceptor`.


In [ ]:
INTERCEPTOR_LOGS = {
    "Module 3a Tools Gateway": "/aws/lambda/agentcore-gateway-request-interceptor",
    "Module 3b AgentCore Gateway": "/aws/lambda/ac-gateway-request-interceptor",
}

events, found = [], []
for path_name, log_group in INTERCEPTOR_LOGS.items():
    try:
        batch = recent_events(log_group, 5, filterPattern='"tools"')
    except logs.exceptions.ResourceNotFoundException:
        print(f"{log_group}: not deployed in this account ({path_name} not taken)")
        continue
    print(f"{log_group}: {len(batch)} interceptor audit events ({path_name})")
    events.extend(batch)
    found.append(path_name)

if found:
    # Print the governance fields this lesson is about. The raw record also
    # carries the tool arguments, which can hold caller-supplied data, and
    # notebook output is saved to disk.
    for e in events:
        try:
            rec = json.loads(e["message"][e["message"].index("{"):])
            print(f"  {rec.get('timestamp', e['timestamp'])}  "
                  f"method={rec.get('method', '?')}  "
                  f"tool={rec.get('tool_name') or '(list)'}  "
                  f"client_id={rec.get('client_id', '?')}")
        except (ValueError, KeyError):
            print(f"  {e['timestamp']}  (unparsed record, {len(e['message'])} chars)")
else:
    print("Neither interceptor log group exists yet -- wire a gateway (connect-gateway-mcp or "
          "connect-gateway-agentcore) and run the agent once, then re-run this cell.")


## Step 3: Open the GenAI Observability Dashboard

Amazon Bedrock ships a managed GenAI Observability view in CloudWatch. Open it to see traces
for every agent invocation — each span shows latency, status, and OpenTelemetry metadata.

Open this URL in your browser, then confirm the region selector (top-right of the console) is
set to the region you deployed into:

```text
https://console.aws.amazon.com/cloudwatch/home#gen-ai-observability
```

## Step 4: Query AgentCore Memory

AgentCore Memory persists conversation history per user/session. Query recent events for the
memory resource attached to the runtime.


In [ ]:
cfn = boto3.client("cloudformation", region_name=REGION)
outputs = {
    o["OutputKey"]: o["OutputValue"]
    for o in cfn.describe_stacks(StackName="FAST-stack")["Stacks"][0]["Outputs"]
}

MEMORY_ARN = outputs.get("MemoryArn", "")
if not MEMORY_ARN:
    print("FAST-stack has no MemoryArn output — check the deployment")
else:
    MEMORY_ID = MEMORY_ARN.rsplit("/", 1)[-1]
    print(f"Memory ID: {MEMORY_ID}")

    # Memory events live on the bedrock-agentcore DATA plane (not the
    # bedrock-agentcore-control plane). ListEvents requires memoryId +
    # sessionId + actorId, so we first list actors, then iterate sessions
    # per actor, then fetch a handful of recent events.
    agentcore_data = boto3.client("bedrock-agentcore", region_name=REGION)

    actors = agentcore_data.list_actors(memoryId=MEMORY_ID, maxResults=10).get("actorSummaries", [])
    print(f"Actors with memory: {len(actors)}")
    if not actors:
        print("No memory yet — send a message through the Amplify UI first.")
    else:
        for a in actors[:3]:
            actor_id = a["actorId"]
            sessions = agentcore_data.list_sessions(
                memoryId=MEMORY_ID, actorId=actor_id, maxResults=3,
            ).get("sessionSummaries", [])
            print(f"  actor={actor_id}  sessions={len(sessions)}")
            for s in sessions[:1]:
                session_id = s["sessionId"]
                events = agentcore_data.list_events(
                    memoryId=MEMORY_ID,
                    actorId=actor_id,
                    sessionId=session_id,
                    maxResults=5,
                ).get("events", [])
                print(f"    session={session_id[:12]}  events={len(events)}")
                for ev in events[:3]:
                    print(f"      {ev.get('eventTimestamp', '?')}  {ev.get('eventId', '?')[:12]}")


## What You Observed

| Surface | Shows | Who Uses It |
|---------|-------|-------------|
| Runtime logs | Model invocations, tool calls, errors | AI/ML engineers debugging agent behavior |
| GenAI Observability | End-to-end traces with latency and spans | Platform teams monitoring production |
| Gateway audit log | Every tool invocation with caller identity | Platform teams tracking tool usage |
| AgentCore Memory | Conversation history per user/session | AI/ML engineers tuning memory strategies |

Proceed to **notebook 07 — Register the Agent (optional)** or skip straight to **notebook 08
— Cleanup**.
